# Infrastructure Forensics: Road Damage CNN (YOLO on RDD2022)
Trains a YOLO object detector on RDD2022 (road images from India, Japan, Czech Republic, Norway, US, China).

**Before running:** Settings → Accelerator = **GPU T4 x2**, Internet = **On**, and the **RDD 2022** dataset added as input.
Run with **Save Version → Save & Run All (Commit)** so it trains in the background.

## 1. Settings

In [ ]:
TRAIN_HOURS = 8          # start ~1 am, results ready by ~10 am
IMG_SIZE    = 640
BATCH       = 32
PRIMARY_WEIGHTS  = 'yolo26s.pt'   # newest Ultralytics model
FALLBACK_WEIGHTS = 'yolo11s.pt'   # used automatically if the first can't load
DEFAULT_NAMES = ['longitudinal crack', 'transverse crack', 'alligator crack', 'other corruption', 'pothole']
OUT = '/kaggle/working/infra_forensics_cnn'

## 2. Install

In [ ]:
!pip install -q -U ultralytics
import os, glob, json, random, shutil, yaml, collections
import ultralytics, torch
from ultralytics import YOLO
print('ultralytics', ultralytics.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on GPU!')
os.makedirs(OUT, exist_ok=True)

# Fail in the first minute, not hours later, if the GPU is missing or unsupported
assert torch.cuda.is_available(), 'No GPU found: Settings > Accelerator > GPU T4 x2'
try:
    (torch.ones(8, device='cuda') * 2).sum().item()
except Exception as e:
    raise RuntimeError('This GPU is not supported by the installed PyTorch. Switch Accelerator to GPU T4 x2. ' + str(e))
print('GPU check passed')

## 3. Find the dataset automatically

In [ ]:
ROOT = '/kaggle/input'
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}

def split_of(path):
    parts = [p.lower() for p in path.split(os.sep)]
    for p in reversed(parts):
        if p.startswith('train'): return 'train'
        if p.startswith('val'):   return 'val'
        if p.startswith('test'):  return 'test'
    return None

found = collections.defaultdict(list)
existing_yaml = None
for dirpath, dirnames, filenames in os.walk(ROOT):
    for f in filenames:
        if f.endswith(('.yaml', '.yml')) and existing_yaml is None:
            existing_yaml = os.path.join(dirpath, f)
    if f'{os.sep}images' not in dirpath:
        continue
    imgs = [f for f in filenames if os.path.splitext(f)[1].lower() in IMG_EXT]
    if not imgs:
        continue
    i = dirpath.rfind(f'{os.sep}images')
    label_dir = dirpath[:i] + f'{os.sep}labels' + dirpath[i+len(os.sep+'images'):]
    if not os.path.isdir(label_dir):
        continue
    s = split_of(dirpath)
    if s:
        found[s].append((dirpath, len(imgs)))

for s, dirs in found.items():
    print(s.upper(), sum(n for _, n in dirs), 'images in', [d for d, _ in dirs])
assert found.get('train'), 'No train/images + train/labels found. Check that the RDD 2022 dataset is added as input.'

# If the dataset has no labelled validation split, use its test split; if none, carve 10% of train
if not found.get('val') and found.get('test'):
    found['val'] = found['test']; print('No val split -> using test split as validation')

## 4. Class names and label statistics

In [ ]:
names = DEFAULT_NAMES
if existing_yaml:
    try:
        y = yaml.safe_load(open(existing_yaml))
        n = y.get('names')
        if isinstance(n, dict): n = [n[k] for k in sorted(n)]
        if n: names = list(n)
        print('Class names from', existing_yaml)
    except Exception as e:
        print('Could not read dataset yaml, using defaults:', e)

def label_stats(img_dirs):
    counts = collections.Counter(); files = 0
    for d, _ in img_dirs:
        i = d.rfind(f'{os.sep}images')
        ld = d[:i] + f'{os.sep}labels' + d[i+len(os.sep+'images'):]
        for lf in glob.glob(os.path.join(ld, '*.txt')):
            files += 1
            for line in open(lf):
                if line.strip(): counts[int(float(line.split()[0]))] += 1
    return files, counts

files, counts = label_stats(found['train'])
max_id = max(counts) if counts else len(names) - 1
if max_id >= len(names):
    names = names + [f'class_{k}' for k in range(len(names), max_id + 1)]
print(f'{files} train label files')
for k in sorted(counts): print(f'  {k} {names[k]:<20} {counts[k]} boxes')
json.dump({'names': names, 'train_box_counts': {names[k]: counts[k] for k in counts}},
          open(f'{OUT}/dataset_stats.json', 'w'), indent=2)

## 5. Build data.yaml

In [ ]:
if found.get('val'):
    train_src = [d for d, _ in found['train']]
    val_src   = [d for d, _ in found['val']]
else:
    all_imgs = sorted(p for d, _ in found['train'] for p in glob.glob(os.path.join(d, '*'))
                      if os.path.splitext(p)[1].lower() in IMG_EXT)
    random.Random(0).shuffle(all_imgs)
    k = int(0.1 * len(all_imgs))
    open(f'{OUT}/val.txt', 'w').write('\n'.join(all_imgs[:k]))
    open(f'{OUT}/train.txt', 'w').write('\n'.join(all_imgs[k:]))
    train_src, val_src = f'{OUT}/train.txt', f'{OUT}/val.txt'
    print('No val split -> carved 10% of train')

DATA_YAML = f'{OUT}/data.yaml'
yaml.safe_dump({'train': train_src, 'val': val_src, 'names': {i: n for i, n in enumerate(names)}},
               open(DATA_YAML, 'w'), sort_keys=False)
print(open(DATA_YAML).read())

## 6. Train (runs for up to TRAIN_HOURS)

In [ ]:
try:
    model = YOLO(PRIMARY_WEIGHTS); used = PRIMARY_WEIGHTS
except Exception as e:
    print('Primary weights unavailable -> fallback:', e)
    model = YOLO(FALLBACK_WEIGHTS); used = FALLBACK_WEIGHTS
print('Base model:', used)

model.train(data=DATA_YAML, epochs=150, time=TRAIN_HOURS, imgsz=IMG_SIZE, batch=BATCH,
            patience=25, workers=4, device=0, seed=0, plots=True,
            project=f'{OUT}/runs', name='rdd2022', exist_ok=True)
RUN = f'{OUT}/runs/rdd2022'
BEST = f'{RUN}/weights/best.pt'
print('Best weights:', BEST, os.path.exists(BEST))

## 7. Evaluate: all countries

In [ ]:
best = YOLO(BEST)
r = best.val(data=DATA_YAML, imgsz=IMG_SIZE, batch=BATCH, plots=True,
             project=f'{OUT}/runs', name='val_all', exist_ok=True)

def summarize(r, tag):
    per_class = {names[int(c)]: round(float(ap), 4) for c, ap in zip(r.box.ap_class_index, r.box.ap50)}
    s = {'subset': tag, 'base_model': used,
         'precision': round(float(r.box.mp), 4), 'recall': round(float(r.box.mr), 4),
         'mAP50': round(float(r.box.map50), 4), 'mAP50-95': round(float(r.box.map), 4),
         'AP50_per_class': per_class,
         'inference_ms_per_image': round(r.speed['inference'], 2),
         'approx_fps': round(1000 / max(sum(r.speed.values()), 1e-6), 1)}
    print(json.dumps(s, indent=2)); return s

metrics = {'all_countries': summarize(r, 'all countries')}

## 8. Evaluate: Indian roads only

In [ ]:
val_imgs = []
for src in (val_src if isinstance(val_src, list) else []):
    val_imgs += [p for p in glob.glob(os.path.join(src, '*')) if os.path.splitext(p)[1].lower() in IMG_EXT]
if isinstance(val_src, str):
    val_imgs = open(val_src).read().split()
india = [p for p in val_imgs if 'india' in os.path.basename(p).lower()]
print(len(india), 'Indian validation images')

if len(india) >= 50:
    open(f'{OUT}/val_india.txt', 'w').write('\n'.join(india))
    y = yaml.safe_load(open(DATA_YAML)); y['val'] = f'{OUT}/val_india.txt'
    yaml.safe_dump(y, open(f'{OUT}/data_india.yaml', 'w'), sort_keys=False)
    ri = best.val(data=f'{OUT}/data_india.yaml', imgsz=IMG_SIZE, batch=BATCH, plots=True,
                  project=f'{OUT}/runs', name='val_india', exist_ok=True)
    metrics['india_only'] = summarize(ri, 'India only')
else:
    print('Filenames do not identify country -> India-only evaluation skipped')

json.dump(metrics, open(f'{OUT}/metrics.json', 'w'), indent=2)

## 9. Sample predictions (for slides)

In [ ]:
pool = india if len(india) >= 12 else val_imgs
samples = random.Random(1).sample(pool, min(16, len(pool)))
best.predict(samples, conf=0.25, save=True, project=f'{OUT}/runs', name='samples', exist_ok=True, verbose=False)
print('Saved to', f'{OUT}/runs/samples')

## 10. Package results

In [ ]:
shutil.copy(BEST, f'{OUT}/infra_forensics_best.pt')
shutil.make_archive('/kaggle/working/infra_forensics_cnn_results', 'zip', OUT)
print('Download /kaggle/working/infra_forensics_cnn_results.zip from the Output tab')
print(json.dumps(metrics, indent=2))